In [25]:
import pandas as pd
import re

train_path = "./X_train/clinical_train.csv"
test_path = "./X_test/clinical_test.csv"

def load_clinical(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test


In [26]:
train, test = load_clinical(train_path, test_path)

In [ ]:
import pandas as pd
import numpy as np
import re


def load_clinical(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    return train, test



def parse_cytogenetics(value):

    if pd.isna(value):
        value = ""

    value = str(value).strip()

    missing = value == ""
    unknown = (
        missing
        or value.lower() in {"none", "nan", "unknown", "na"}
    )

    s = re.sub(r"\s+", "", value.lower())

    features = {
        "cyto_unknown": int(unknown),
        "cyto_normal": 0,
        "cyto_abnormal": 0,

        "is_mosaic": int("/" in s),

        "n_clones": 1,
        "n_abnormal_clones": 0,
        "abnormal_cell_pct": np.nan,

        "n_cyto_events": 0,

        "is_complex": 0,
        "is_monosomal": 0,

        "has_chr3_abn": 0,
        "has_inv3": 0,
        "has_t_3_3": 0,
        "has_3q21": 0,
        "has_3q26": 0,

        "has_minus5": 0,
        "has_5q_del": 0,

        "has_minus7": 0,

        "has_17p_abn": 0,
        "has_20q_del": 0,

        "has_marker": 0,

        "has_deletion": 0,
        "has_translocation": 0,
        "has_inversion": 0,
        "has_duplication": 0,
        "has_additional_material": 0,
        "has_gain": 0,
        "has_loss": 0,
    }

    # Completely missing / unknown
    if unknown:
        return features

    clones = s.split("/")

    features["n_clones"] = len(clones)
    features["is_mosaic"] = int(len(clones) > 1)

    total_cells = 0
    abnormal_cells = 0

    for clone in clones:

        count_match = re.search(r"\[(\d+)\]", clone)

        if count_match:
            cell_count = int(count_match.group(1))
            total_cells += cell_count
        else:
            cell_count = None

        # Remove clone cell-count annotation
        clone_clean = re.sub(r"\[\d+\]", "", clone)

        # A clone is abnormal if it contains an abnormality
        has_abnormality = bool(
            re.search(
                r"""
                del\(|dup\(|inv\(|
                t\(|dic\(|der\(|
                add\(|i\(|idic\(|
                -[1-9][0-9]*|
                \+[1-9][0-9]*
                """,
                clone_clean,
                flags=re.VERBOSE,
            )
        )

        if has_abnormality:
            features["n_abnormal_clones"] += 1

            if cell_count is not None:
                abnormal_cells += cell_count

    if total_cells > 0:
        features["abnormal_cell_pct"] = (
            100 * abnormal_cells / total_cells
        )

    # --------------------------------------------------------
    # Normal karyotype
    # --------------------------------------------------------

    # Examples:
    # 46,xx
    # 46,xy
    # 46,x...
    #
    # Only classify as normal if there are no explicit abnormal
    # cytogenetic events.

    has_structural_abnormality = bool(
        re.search(
            r"del\(|dup\(|inv\(|t\(|dic\(|der\(|add\(|"
            r"i\(|idic\(|r\(|hsr\(",
            s
        )
    )

    has_numeric_abnormality = bool(
        re.search(
            r"(?<![a-z])-[1-9][0-9]*(?![a-z])|"
            r"(?<![a-z])\+[1-9][0-9]*(?![a-z])",
            s
        )
    )

    has_explicit_abnormality = (
        has_structural_abnormality
        or has_numeric_abnormality
        or "complex/other" in s
        or "mar" in s
    )

    if not has_explicit_abnormality:
        if re.match(r"^(?:\d+[-~]\d+)?46,[xy_]+$", s):
            features["cyto_normal"] = 1
        elif re.match(r"^46,[xy_]+$", s):
            features["cyto_normal"] = 1

    # Everything non-normal and interpretable is abnormal
    if not features["cyto_normal"]:
        features["cyto_abnormal"] = 1

    # --------------------------------------------------------
    # General structural abnormalities
    # --------------------------------------------------------

    features["has_deletion"] = int("del(" in s)
    features["has_translocation"] = int(
        bool(re.search(r"t\(", s))
    )
    features["has_inversion"] = int("inv(" in s)
    features["has_duplication"] = int("dup(" in s)
    features["has_additional_material"] = int("add(" in s)

    # Marker chromosomes
    features["has_marker"] = int(
        bool(re.search(r"(?:\+mar|-?mar|mar\d*)", s))
    )

    # Gains / losses
    features["has_gain"] = int(
        bool(re.search(r"\+[1-9][0-9]*", s))
    )

    features["has_loss"] = int(
        bool(re.search(r"-[1-9][0-9]*", s))
    )

    # --------------------------------------------------------
    # Chromosome-specific numerical abnormalities
    # --------------------------------------------------------

    # Match -7, -5 etc., but not arbitrary numbers inside
    # structural notation.
    def has_chromosome_loss(chromosome):
        return int(
            bool(
                re.search(
                    rf"(?<![\w])-{chromosome}(?![\w])",
                    s
                )
            )
        )

    features["has_minus5"] = has_chromosome_loss(5)
    features["has_minus7"] = has_chromosome_loss(7)

    # --------------------------------------------------------
    # 5q
    # --------------------------------------------------------

    features["has_5q_del"] = int(
        bool(
            re.search(
                r"del\(5\)\([^)]*q",
                s
            )
        )
    )

    # --------------------------------------------------------
    # 17p
    # --------------------------------------------------------

    features["has_17p_abn"] = int(
        bool(
            re.search(
                r"""
                (?:del|add|dup|inv)\(17\)\([^)]*p
                |
                i\(17\)
                |
                idic\(17\)
                """,
                s,
                flags=re.VERBOSE,
            )
        )
    )

    # --------------------------------------------------------
    # 20q
    # --------------------------------------------------------

    features["has_20q_del"] = int(
        bool(
            re.search(
                r"del\(20\)\([^)]*q",
                s
            )
        )
    )

    # --------------------------------------------------------
    # Chromosome 3
    # --------------------------------------------------------

    # Any explicit chromosome-3 structural abnormality
    features["has_chr3_abn"] = int(
        bool(
            re.search(
                r"""
                (?:del|add|dup|inv)\(3\)
                |
                t\(3;
                |
                t\([^;]+;3\)
                |
                der\(3
                |
                dic\(3
                """,
                s,
                flags=re.VERBOSE,
            )
        )
    )

    # inv(3)
    features["has_inv3"] = int(
        bool(
            re.search(
                r"inv\(3\)",
                s
            )
        )
    )

    # t(3;3)
    features["has_t_3_3"] = int(
        bool(
            re.search(
                r"t\(3[;:]3\)",
                s
            )
        )
    )

    # 3q21
    features["has_3q21"] = int(
        bool(
            re.search(
                r"3\)\([^)]*q21|3;3\)\(q21",
                s
            )
        )
    )

    # 3q26
    features["has_3q26"] = int(
        bool(
            re.search(
                r"3\)\([^)]*q26|q26",
                s
            )
        )
    )

    # --------------------------------------------------------
    # Number of cytogenetic events
    # --------------------------------------------------------

    event_patterns = [
        r"del\(",
        r"dup\(",
        r"inv\(",
        r"t\(",
        r"dic\(",
        r"der\(",
        r"add\(",
        r"i\(",
        r"idic\(",
        r"r\(",
        r"hsr\(",
        r"(?<![\w])-[1-9][0-9]*(?![\w])",
        r"(?<![\w])\+[1-9][0-9]*(?![\w])",
    ]

    event_count = 0

    for pattern in event_patterns:
        event_count += len(re.findall(pattern, s))

    features["n_cyto_events"] = event_count

    # --------------------------------------------------------
    # Complex karyotype
    #
    # We use >=3 detected cytogenetic events as the structured
    # definition, while retaining "Complex/Other" explicitly.
    # --------------------------------------------------------

    features["is_complex"] = int(
        event_count >= 3
        or "complex/other" in s
    )

    # --------------------------------------------------------
    # Monosomal pattern
    #
    # A practical feature here is presence of >=2 autosomal
    # chromosome losses. We do not count sex chromosomes.
    # --------------------------------------------------------

    autosomal_losses = re.findall(
        r"(?<![\w])-(\d+)(?![\w])",
        s
    )

    autosomal_losses = [
        int(x)
        for x in autosomal_losses
        if 1 <= int(x) <= 22
    ]

    features["is_monosomal"] = int(
        len(set(autosomal_losses)) >= 2
    )

    return features


# ============================================================
# Add cytogenetic features
# ============================================================

def add_cytogenetic_features(df, cytogenetic_col="CYTOGENETICS"):
    df = df.copy()

    if cytogenetic_col not in df.columns:
        print(
            f"Warning: '{cytogenetic_col}' not found. "
            "No cytogenetic features added."
        )
        return df

    parsed = df[cytogenetic_col].apply(parse_cytogenetics)

    cytogenetic_features = pd.DataFrame(
        parsed.tolist(),
        index=df.index
    )

    # Prevent accidental duplicate columns
    duplicate_columns = [
        col
        for col in cytogenetic_features.columns
        if col in df.columns
    ]

    if duplicate_columns:
        df = df.drop(columns=duplicate_columns)

    df = pd.concat(
        [df, cytogenetic_features],
        axis=1
    )

    return df


# ============================================================
# Clinical cleaning
# ============================================================

def clean_clinical(df):
    """
    Model-agnostic clinical cleaning.

    Do NOT perform scaling, target-based feature selection,
    or model-specific encoding here.
    """

    df = df.copy()

    # --------------------------------------------------------
    # Standardise obvious missing values
    # --------------------------------------------------------

    missing_values = [
        "",
        " ",
        "NA",
        "N/A",
        "na",
        "n/a",
        "None",
        "none",
        "NULL",
        "null",
        "?",
    ]

    df = df.replace(
        missing_values,
        np.nan
    )

    # --------------------------------------------------------
    # Convert numeric-looking columns where appropriate
    # --------------------------------------------------------

    for col in df.columns:

        if col in {"ID", "CYTOGENETICS"}:
            continue

        # Don't blindly convert everything. Only convert object
        # columns where almost all non-null values are numeric.
        if df[col].dtype == "object":

            converted = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            non_null = df[col].notna().sum()

            if non_null > 0:
                conversion_rate = (
                    converted.notna().sum() / non_null
                )

                if conversion_rate >= 0.95:
                    df[col] = converted

    return df


# ============================================================
# Align train / test
# ============================================================

def align_train_test(train, test):
    """
    Ensure train and test contain the same feature columns.

    We deliberately do not fit an encoder here. This function only
    guarantees the same schema.
    """

    train = train.copy()
    test = test.copy()

    # Columns present in only one dataset
    train_only = set(train.columns) - set(test.columns)
    test_only = set(test.columns) - set(train.columns)

    # For model-agnostic preprocessing, retain columns that exist
    # in both datasets.
    common_columns = [
        col for col in train.columns
        if col in test.columns
    ]

    train = train[common_columns]
    test = test[common_columns]

    return train, test


# ============================================================
# Main preprocessing function
# ============================================================

def preprocess_clinical(
    train_df,
    test_df,
    cytogenetic_col="CYTOGENETICS"
):

    train = train_df.copy()
    test = test_df.copy()

    # --------------------------------------------------------
    # 1. Clinical cleaning
    # --------------------------------------------------------

    train = clean_clinical(train)
    test = clean_clinical(test)

    # --------------------------------------------------------
    # 2. Cytogenetic feature engineering
    # --------------------------------------------------------

    train = add_cytogenetic_features(
        train,
        cytogenetic_col=cytogenetic_col
    )

    test = add_cytogenetic_features(
        test,
        cytogenetic_col=cytogenetic_col
    )

    # --------------------------------------------------------
    # 3. Make train/test schema identical
    # --------------------------------------------------------

    train, test = align_train_test(
        train,
        test
    )

    return train, test


# ============================================================
# Example usage
# ============================================================

train_path = "./X_train/clinical_train.csv"
test_path = "./X_test/clinical_test.csv"

train, test = load_clinical(
    train_path,
    test_path
)

clinical_train, clinical_test = preprocess_clinical(
    train,
    test
)

print("Train shape:", clinical_train.shape)
print("Test shape:", clinical_test.shape)

print("\nCytogenetic features:")
print([
    col for col in clinical_train.columns
    if col.startswith(("cyto_", "has_", "is_", "n_", "abnormal_"))
])

Train shape: (3323, 37)
Test shape: (1193, 37)

Cytogenetic features:
['cyto_unknown', 'cyto_normal', 'cyto_abnormal', 'is_mosaic', 'n_clones', 'n_abnormal_clones', 'abnormal_cell_pct', 'n_cyto_events', 'is_complex', 'is_monosomal', 'has_chr3_abn', 'has_inv3', 'has_t_3_3', 'has_3q21', 'has_3q26', 'has_minus5', 'has_5q_del', 'has_minus7', 'has_17p_abn', 'has_20q_del', 'has_marker', 'has_deletion', 'has_translocation', 'has_inversion', 'has_duplication', 'has_additional_material', 'has_gain', 'has_loss']


In [28]:
# ------------------------------------------------------------
# 1. Cytogenetic state coverage
# ------------------------------------------------------------

state_cols = [
    "cyto_unknown",
    "cyto_normal",
    "cyto_abnormal",
]

state_sum = clinical_train[state_cols].sum(axis=1)

print(
    "Exactly one state:",
    (state_sum == 1).sum()
)

print(
    "Zero states:",
    (state_sum == 0).sum()
)

print(
    "Multiple states:",
    (state_sum > 1).sum()
)

Exactly one state: 3323
Zero states: 0
Multiple states: 0


In [29]:
cyto_cols = [
    "is_mosaic",
    "is_complex",
    "is_monosomal",
    "has_chr3_abn",
    "has_inv3",
    "has_t_3_3",
    "has_3q21",
    "has_3q26",
    "has_minus5",
    "has_5q_del",
    "has_minus7",
    "has_17p_abn",
    "has_20q_del",
    "has_marker",
    "has_deletion",
    "has_translocation",
    "has_inversion",
    "has_duplication",
    "has_additional_material",
]

print(
    clinical_train[cyto_cols]
    .sum()
    .sort_values(ascending=False)
)

is_mosaic                  860
has_deletion               603
is_complex                 329
has_5q_del                 321
has_translocation          190
has_minus7                 168
has_marker                 160
has_additional_material    152
is_monosomal               148
has_20q_del                107
has_chr3_abn                93
has_minus5                  59
has_inversion               44
has_17p_abn                 37
has_3q26                    33
has_3q21                    23
has_duplication             14
has_inv3                    11
has_t_3_3                    6
dtype: int64


In [30]:
chr3_mask = clinical_train["has_chr3_abn"] == 1

print(
    pd.DataFrame({
        "raw": train.loc[chr3_mask, "CYTOGENETICS"],
        "inv3": clinical_train.loc[chr3_mask, "has_inv3"],
        "t_3_3": clinical_train.loc[chr3_mask, "has_t_3_3"],
        "3q21": clinical_train.loc[chr3_mask, "has_3q21"],
        "3q26": clinical_train.loc[chr3_mask, "has_3q26"],
    }).to_string(index=False)
)

                                                                                                                                                                                                                                                                                                           raw  inv3  t_3_3  3q21  3q26
                                                                                                                                                                                                                                                                            46,xy,t(3;3)(q25;q27)[8]/46,xy[12]     0      1     0     0
                                                                                                                                                                                                                                                                             46,xy,del(3)(q26q27)[15]/46,xy[5]     0      0     0     1
                

In [39]:
clinical_train_to_merge = clinical_train.copy().drop(columns=["CYTOGENETICS"])
clinical_test_to_merge = clinical_test.copy().drop(columns=["CYTOGENETICS"])